<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/notebooks/Model03_Previous_Application_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

Mounted at /content/drive


In [2]:
DATA_PATH = "/content/drive/MyDrive/datasets/raw/"
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"
os.makedirs(RESULTS_PATH, exist_ok=True)

In [3]:
application = pd.read_csv(DATA_PATH + "application_train.csv")
prev = pd.read_csv(DATA_PATH + "previous_application.csv")

print("Application shape:", application.shape)
print("Previous application shape:", prev.shape)

Application shape: (307511, 122)
Previous application shape: (1670214, 37)


In [4]:
# MODEL02 cleaning + features on application
# DAYS_EMPLOYED anomaly
application["DAYS_EMPLOYED_ANOM"] = (application["DAYS_EMPLOYED"] == 365243).astype(int)
application["DAYS_EMPLOYED"] = application["DAYS_EMPLOYED"].replace(365243, np.nan)

# EXT_SOURCE missingness
application["EXT_SOURCE_1_MISSING"] = application["EXT_SOURCE_1"].isna().astype(int)
application["EXT_SOURCE_3_MISSING"] = application["EXT_SOURCE_3"].isna().astype(int)

# Age
application["AGE_YEARS"] = -application["DAYS_BIRTH"] / 365.25

# Employment
application["EMPLOYMENT_YEARS"] = -application["DAYS_EMPLOYED"] / 365.25

# Financial ratios
application["CREDIT_INCOME_RATIO"] = application["AMT_CREDIT"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_INCOME_RATIO"] = application["AMT_ANNUITY"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_CREDIT_RATIO"] = application["AMT_ANNUITY"] / application["AMT_CREDIT"]
application["GOODS_CREDIT_RATIO"] = application["AMT_GOODS_PRICE"] / application["AMT_CREDIT"]

# Employment / age
application["EMPLOYMENT_AGE_RATIO"] = application["EMPLOYMENT_YEARS"] / application["AGE_YEARS"]

# EXT_SOURCE summary
ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
application["EXT_SOURCE_MEAN"] = application[ext_cols].mean(axis=1)
application["EXT_SOURCE_MIN"] = application[ext_cols].min(axis=1)
application["EXT_SOURCE_MAX"] = application[ext_cols].max(axis=1)
application["EXT_SOURCE_STD"] = application[ext_cols].std(axis=1)

# EXT_SOURCE interactions
application["EXT_SOURCE_1_2"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_2"]
application["EXT_SOURCE_1_3"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_3"]
application["EXT_SOURCE_2_3"] = application["EXT_SOURCE_2"] * application["EXT_SOURCE_3"]

print("MODEL02 application features recreated.")
print("Application shape:", application.shape)

MODEL02 application features recreated.
Application shape: (307511, 139)


In [5]:
print("Application unique applicants:", application["SK_ID_CURR"].nunique())
print("Previous application unique applicants:", prev["SK_ID_CURR"].nunique())
print("Previous application rows:", len(prev))

Application unique applicants: 307511
Previous application unique applicants: 338857
Previous application rows: 1670214


In [6]:
sentinel_cols = ['DAYS_FIRST_DRAWING', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']

for col in sentinel_cols:
    prev[col] = prev[col].replace(365243, np.nan)

for col in sentinel_cols:
    print(col, "- remaining 365243:", (prev[col] == 365243).sum())

DAYS_FIRST_DRAWING - remaining 365243: 0
DAYS_LAST_DUE_1ST_VERSION - remaining 365243: 0
DAYS_LAST_DUE - remaining 365243: 0
DAYS_TERMINATION - remaining 365243: 0


In [7]:
prev_count = (
    prev.groupby("SK_ID_CURR")
    .size()
    .rename("PREV_APPLICATION_COUNT")
    .reset_index()
)

display(prev_count.head())

,SK_ID_CURR,PREV_APPLICATION_COUNT
0,100001,1
1,100002,1
2,100003,3
3,100004,1
4,100005,2


In [8]:
financial_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "AMT_CREDIT": ["mean", "max", "sum"],
        "AMT_APPLICATION": ["mean", "max", "sum"],
        "AMT_ANNUITY": ["mean", "max", "sum"],
        "AMT_GOODS_PRICE": ["mean", "max", "sum"],
        "AMT_DOWN_PAYMENT": ["mean", "max"],
        "RATE_DOWN_PAYMENT": ["mean", "max"]
    })
)

financial_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in financial_agg.columns]
financial_agg = financial_agg.reset_index()

display(financial_agg.head())

,SK_ID_CURR,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_CREDIT_SUM,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_APPLICATION_SUM,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,PREV_AMT_ANNUITY_SUM,PREV_AMT_GOODS_PRICE_MEAN,PREV_AMT_GOODS_PRICE_MAX,PREV_AMT_GOODS_PRICE_SUM,PREV_AMT_DOWN_PAYMENT_MEAN,PREV_AMT_DOWN_PAYMENT_MAX,PREV_RATE_DOWN_PAYMENT_MEAN,PREV_RATE_DOWN_PAYMENT_MAX
0,100001,23787.00,23787.0,23787.0,24835.50,24835.5,24835.5,3951.000,3951.000,3951.000,24835.5,24835.5,24835.5,2520.0,2520.0,0.104326,0.104326
1,100002,179055.00,179055.0,179055.0,179055.00,179055.0,179055.0,9251.775,9251.775,9251.775,179055.0,179055.0,179055.0,0.0,0.0,0.000000,0.000000
2,100003,484191.00,1035882.0,1452573.0,435436.50,900000.0,1306309.5,56553.990,98356.995,169661.970,435436.5,900000.0,1306309.5,3442.5,6885.0,0.050030,0.100061
3,100004,20106.00,20106.0,20106.0,24282.00,24282.0,24282.0,5357.250,5357.250,5357.250,24282.0,24282.0,24282.0,4860.0,4860.0,0.212008,0.212008
4,100005,20076.75,40153.5,40153.5,22308.75,44617.5,44617.5,4813.200,4813.200,4813.200,44617.5,44617.5,44617.5,4464.0,4464.0,0.108964,0.108964


In [9]:
#Credit/application relationship features

prev["PREV_CREDIT_APPL_RATIO"] = prev["AMT_CREDIT"] / prev["AMT_APPLICATION"].replace(0, np.nan)
prev["PREV_CREDIT_APPL_DIFF"] = prev["AMT_CREDIT"] - prev["AMT_APPLICATION"]

print("Infinite values in ratio:", np.isinf(prev["PREV_CREDIT_APPL_RATIO"]).sum())
print("NaN values in ratio:", prev["PREV_CREDIT_APPL_RATIO"].isna().sum())

Infinite values in ratio: 0
NaN values in ratio: 392402


In [10]:
#Aggregate relationship features

relationship_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_CREDIT_APPL_RATIO": ["mean", "max"],
        "PREV_CREDIT_APPL_DIFF": ["mean", "max"]
    })
)

relationship_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in relationship_agg.columns]
relationship_agg = relationship_agg.reset_index()

display(relationship_agg.head())

,SK_ID_CURR,PREV_PREV_CREDIT_APPL_RATIO_MEAN,PREV_PREV_CREDIT_APPL_RATIO_MAX,PREV_PREV_CREDIT_APPL_DIFF_MEAN,PREV_PREV_CREDIT_APPL_DIFF_MAX
0,100001,0.957782,0.957782,-1048.5,-1048.5
1,100002,1.000000,1.000000,0.0,0.0
2,100003,1.057664,1.150980,48754.5,135882.0
3,100004,0.828021,0.828021,-4176.0,-4176.0
4,100005,0.899950,0.899950,-2232.0,0.0


In [11]:
#Contract status flags

print(prev["NAME_CONTRACT_STATUS"].value_counts(dropna=False))

prev["PREV_APPROVED"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
prev["PREV_REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev["PREV_CANCELED"] = (prev["NAME_CONTRACT_STATUS"] == "Canceled").astype(int)
prev["PREV_UNUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Unused offer").astype(int)

NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64


In [12]:
#Status counts

status_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_APPROVED": "sum",
        "PREV_REFUSED": "sum",
        "PREV_CANCELED": "sum",
        "PREV_UNUSED": "sum"
    })
    .reset_index()
)

status_agg = status_agg.rename(columns={
    "PREV_APPROVED": "PREV_APPROVED_COUNT",
    "PREV_REFUSED": "PREV_REFUSED_COUNT",
    "PREV_CANCELED": "PREV_CANCELED_COUNT",
    "PREV_UNUSED": "PREV_UNUSED_COUNT"
})

display(status_agg.head())

,SK_ID_CURR,PREV_APPROVED_COUNT,PREV_REFUSED_COUNT,PREV_CANCELED_COUNT,PREV_UNUSED_COUNT
0,100001,1,0,0,0
1,100002,1,0,0,0
2,100003,3,0,0,0
3,100004,1,0,0,0
4,100005,1,0,1,0


In [13]:
#Status rates (fixed)

status_agg = status_agg.merge(
    prev_count[["SK_ID_CURR", "PREV_APPLICATION_COUNT"]],
    on="SK_ID_CURR",
    how="left"
)

status_agg["PREV_APPROVAL_RATE"] = status_agg["PREV_APPROVED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_REFUSAL_RATE"] = status_agg["PREV_REFUSED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_CANCELLATION_RATE"] = status_agg["PREV_CANCELED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]

status_agg = status_agg.drop(columns=["PREV_APPLICATION_COUNT"])

display(status_agg.head())

,SK_ID_CURR,PREV_APPROVED_COUNT,PREV_REFUSED_COUNT,PREV_CANCELED_COUNT,PREV_UNUSED_COUNT,PREV_APPROVAL_RATE,PREV_REFUSAL_RATE,PREV_CANCELLATION_RATE
0,100001,1,0,0,0,1.0,0.0,0.0
1,100002,1,0,0,0,1.0,0.0,0.0
2,100003,3,0,0,0,1.0,0.0,0.0
3,100004,1,0,0,0,1.0,0.0,0.0
4,100005,1,0,1,0,0.5,0.0,0.5


In [14]:
#DAYS_DECISION aggregation

decision_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"DAYS_DECISION": ["min", "max", "mean"]})
)

decision_agg.columns = ["PREV_DAYS_DECISION_" + col[1].upper() for col in decision_agg.columns]
decision_agg = decision_agg.reset_index()

display(decision_agg.head())

,SK_ID_CURR,PREV_DAYS_DECISION_MIN,PREV_DAYS_DECISION_MAX,PREV_DAYS_DECISION_MEAN
0,100001,-1740,-1740,-1740.0
1,100002,-606,-606,-606.0
2,100003,-2341,-746,-1305.0
3,100004,-815,-815,-815.0
4,100005,-757,-315,-536.0


In [15]:
#CNT_PAYMENT aggregation

payment_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"CNT_PAYMENT": ["mean", "max", "sum"]})
)

payment_agg.columns = ["PREV_CNT_PAYMENT_" + col[1].upper() for col in payment_agg.columns]
payment_agg = payment_agg.reset_index()

display(payment_agg.head())

,SK_ID_CURR,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX,PREV_CNT_PAYMENT_SUM
0,100001,8.0,8.0,8.0
1,100002,24.0,24.0,24.0
2,100003,10.0,12.0,30.0
3,100004,4.0,4.0,4.0
4,100005,12.0,12.0,12.0


In [16]:
#Merge all previous-application features

prev_features = prev_count.copy()
prev_features = prev_features.merge(financial_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(relationship_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(status_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(decision_agg, on="SK_ID_CURR", how="left")
prev_features = prev_features.merge(payment_agg, on="SK_ID_CURR", how="left")

print("Previous-application feature table shape:", prev_features.shape)
display(prev_features.head())

Previous-application feature table shape: (338857, 35)


,SK_ID_CURR,PREV_APPLICATION_COUNT,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_CREDIT_SUM,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_APPLICATION_SUM,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,...,PREV_UNUSED_COUNT,PREV_APPROVAL_RATE,PREV_REFUSAL_RATE,PREV_CANCELLATION_RATE,PREV_DAYS_DECISION_MIN,PREV_DAYS_DECISION_MAX,PREV_DAYS_DECISION_MEAN,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX,PREV_CNT_PAYMENT_SUM
0,100001,1,23787.00,23787.0,23787.0,24835.50,24835.5,24835.5,3951.000,3951.000,...,0,1.0,0.0,0.0,-1740,-1740,-1740.0,8.0,8.0,8.0
1,100002,1,179055.00,179055.0,179055.0,179055.00,179055.0,179055.0,9251.775,9251.775,...,0,1.0,0.0,0.0,-606,-606,-606.0,24.0,24.0,24.0
2,100003,3,484191.00,1035882.0,1452573.0,435436.50,900000.0,1306309.5,56553.990,98356.995,...,0,1.0,0.0,0.0,-2341,-746,-1305.0,10.0,12.0,30.0
3,100004,1,20106.00,20106.0,20106.0,24282.00,24282.0,24282.0,5357.250,5357.250,...,0,1.0,0.0,0.0,-815,-815,-815.0,4.0,4.0,4.0
4,100005,2,20076.75,40153.5,40153.5,22308.75,44617.5,44617.5,4813.200,4813.200,...,0,0.5,0.0,0.5,-757,-315,-536.0,12.0,12.0,12.0


In [17]:
print("Rows:", len(prev_features))
print("Unique applicants:", prev_features["SK_ID_CURR"].nunique())
print("One row per applicant:", len(prev_features) == prev_features["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", prev_features["SK_ID_CURR"].duplicated().sum())

Rows: 338857
Unique applicants: 338857
One row per applicant: True
Duplicate applicant IDs: 0


In [19]:
#feature list
prev_feature_cols = [col for col in prev_features.columns if col != "SK_ID_CURR"]

print("Number of previous-application features:", len(prev_feature_cols))
for col in prev_feature_cols:
    print("-", col)

Number of previous-application features: 34
- PREV_APPLICATION_COUNT
- PREV_AMT_CREDIT_MEAN
- PREV_AMT_CREDIT_MAX
- PREV_AMT_CREDIT_SUM
- PREV_AMT_APPLICATION_MEAN
- PREV_AMT_APPLICATION_MAX
- PREV_AMT_APPLICATION_SUM
- PREV_AMT_ANNUITY_MEAN
- PREV_AMT_ANNUITY_MAX
- PREV_AMT_ANNUITY_SUM
- PREV_AMT_GOODS_PRICE_MEAN
- PREV_AMT_GOODS_PRICE_MAX
- PREV_AMT_GOODS_PRICE_SUM
- PREV_AMT_DOWN_PAYMENT_MEAN
- PREV_AMT_DOWN_PAYMENT_MAX
- PREV_RATE_DOWN_PAYMENT_MEAN
- PREV_RATE_DOWN_PAYMENT_MAX
- PREV_PREV_CREDIT_APPL_RATIO_MEAN
- PREV_PREV_CREDIT_APPL_RATIO_MAX
- PREV_PREV_CREDIT_APPL_DIFF_MEAN
- PREV_PREV_CREDIT_APPL_DIFF_MAX
- PREV_APPROVED_COUNT
- PREV_REFUSED_COUNT
- PREV_CANCELED_COUNT
- PREV_UNUSED_COUNT
- PREV_APPROVAL_RATE
- PREV_REFUSAL_RATE
- PREV_CANCELLATION_RATE
- PREV_DAYS_DECISION_MIN
- PREV_DAYS_DECISION_MAX
- PREV_DAYS_DECISION_MEAN
- PREV_CNT_PAYMENT_MEAN
- PREV_CNT_PAYMENT_MAX
- PREV_CNT_PAYMENT_SUM


In [20]:
#Merge with MODEL02-enhanced application data

application_model = application.copy()

print("Application shape before merge:", application_model.shape)

application_model = application_model.merge(prev_features, on="SK_ID_CURR", how="left")

print("Application shape after merge:", application_model.shape)

Application shape before merge: (307511, 139)
Application shape after merge: (307511, 173)


In [21]:
#applicants with no history
no_history = application_model["PREV_APPLICATION_COUNT"].isna().sum()

print("Applicants without previous history:", no_history)
print("Percentage:", no_history / len(application_model) * 100)

Applicants without previous history: 16454
Percentage: 5.350702901684818


In [22]:
application_model["HAS_PREVIOUS_APPLICATION"] = application_model["PREV_APPLICATION_COUNT"].notna().astype(int)
application_model["PREV_APPLICATION_COUNT"] = application_model["PREV_APPLICATION_COUNT"].fillna(0)

print(application_model["HAS_PREVIOUS_APPLICATION"].value_counts())

HAS_PREVIOUS_APPLICATION
1    291057
0     16454
Name: count, dtype: int64


In [23]:
print("Final application shape:", application_model.shape)
print("Unique applicants:", application_model["SK_ID_CURR"].nunique())
print("Duplicate applicants:", application_model["SK_ID_CURR"].duplicated().sum())

Final application shape: (307511, 174)
Unique applicants: 307511
Duplicate applicants: 0


In [24]:
X = application_model.drop(columns=["TARGET", "SK_ID_CURR"])
y = application_model["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 172)
y shape: (307511,)


In [25]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

Training shape: (246008, 172)
Validation shape: (61503, 172)

Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Validation target distribution:
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [26]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 156
Categorical features: 16


In [27]:
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [28]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [29]:
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

In [30]:
print("Training XGBoost with MODEL02 + previous-application features...")
xgb_pipeline.fit(X_train, y_train)
print("Training complete.")

Training XGBoost with MODEL02 + previous-application features...
Training complete.


In [31]:
valid_proba = xgb_pipeline.predict_proba(X_valid)[:, 1]
print("Predictions generated.")

Predictions generated.


In [32]:
roc_auc = roc_auc_score(y_valid, valid_proba)
pr_auc = average_precision_score(y_valid, valid_proba)

print("XGBOOST + MODEL02 + PREVIOUS APPLICATION FEATURES")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

XGBOOST + MODEL02 + PREVIOUS APPLICATION FEATURES
ROC-AUC: 0.7754
PR-AUC:  0.2659


In [33]:
MODEL02_ROC_AUC = 0.769403
MODEL02_PR_AUC = 0.262725

roc_change = roc_auc - MODEL02_ROC_AUC
pr_change = pr_auc - MODEL02_PR_AUC

print("IMPROVEMENT OVER MODEL02")
print(f"ROC-AUC change: {roc_change:+.4f}")
print(f"PR-AUC change:  {pr_change:+.4f}")

IMPROVEMENT OVER MODEL02
ROC-AUC change: +0.0060
PR-AUC change:  +0.0031


In [34]:
previous_application_result = pd.DataFrame({
    "Experiment": ["XGBoost + MODEL02 + Previous Application Features"],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc],
    "ROC-AUC Change": [roc_change],
    "PR-AUC Change": [pr_change]
})

display(previous_application_result.style.format({
    "ROC-AUC": "{:.4f}",
    "PR-AUC": "{:.4f}",
    "ROC-AUC Change": "{:+.4f}",
    "PR-AUC Change": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change,PR-AUC Change
0,XGBoost + MODEL02 + Previous Application Features,0.7754,0.2659,+0.0060,+0.0031


In [35]:
previous_application_result.to_csv(RESULTS_PATH + "previous_application_experiment.csv", index=False)
print("Experiment saved to:", RESULTS_PATH + "previous_application_experiment.csv")

Experiment saved to: /content/drive/MyDrive/RupeeRisk/previous_application_experiment.csv


In [37]:
!pip install mlflow -q
import mlflow
mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db")
mlflow.set_experiment("RupeeRisk")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132

<Experiment: artifact_location='/content/mlruns/1', creation_time=1787393296537, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787393296537, lifecycle_stage='active', name='RupeeRisk', tags={}, trace_location=None, workspace='default'>

In [38]:
with mlflow.start_run(run_name="XGBoost_Previous_Application"):
    mlflow.log_param("stage", "MODEL03 - Previous Application Feature Engineering")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("builds_on", "MODEL02 application features")
    mlflow.log_param("n_previous_application_features", len(prev_feature_cols))
    mlflow.log_param("scale_pos_weight", False)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc_change_vs_MODEL02", roc_change)
    mlflow.log_metric("pr_auc_change_vs_MODEL02", pr_change)

print("MODEL03 logged to MLflow.")

MODEL03 logged to MLflow.


In [39]:
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]])

,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Previous_Application,0.775428,0.265853
1,XGBoost_Application_Features,0.769403,0.262725
2,XGBoost_scale_pos_weight,0.760000,0.249300
3,XGBoost_Baseline,0.761200,0.251600
4,Logistic_Regression_Baseline,0.750100,0.232600


In [40]:
print("Previous application features created:")
for col in prev_feature_cols:
    print("-", col)
print("\nTotal features:", len(prev_feature_cols))

Previous application features created:
- PREV_APPLICATION_COUNT
- PREV_AMT_CREDIT_MEAN
- PREV_AMT_CREDIT_MAX
- PREV_AMT_CREDIT_SUM
- PREV_AMT_APPLICATION_MEAN
- PREV_AMT_APPLICATION_MAX
- PREV_AMT_APPLICATION_SUM
- PREV_AMT_ANNUITY_MEAN
- PREV_AMT_ANNUITY_MAX
- PREV_AMT_ANNUITY_SUM
- PREV_AMT_GOODS_PRICE_MEAN
- PREV_AMT_GOODS_PRICE_MAX
- PREV_AMT_GOODS_PRICE_SUM
- PREV_AMT_DOWN_PAYMENT_MEAN
- PREV_AMT_DOWN_PAYMENT_MAX
- PREV_RATE_DOWN_PAYMENT_MEAN
- PREV_RATE_DOWN_PAYMENT_MAX
- PREV_PREV_CREDIT_APPL_RATIO_MEAN
- PREV_PREV_CREDIT_APPL_RATIO_MAX
- PREV_PREV_CREDIT_APPL_DIFF_MEAN
- PREV_PREV_CREDIT_APPL_DIFF_MAX
- PREV_APPROVED_COUNT
- PREV_REFUSED_COUNT
- PREV_CANCELED_COUNT
- PREV_UNUSED_COUNT
- PREV_APPROVAL_RATE
- PREV_REFUSAL_RATE
- PREV_CANCELLATION_RATE
- PREV_DAYS_DECISION_MIN
- PREV_DAYS_DECISION_MAX
- PREV_DAYS_DECISION_MEAN
- PREV_CNT_PAYMENT_MEAN
- PREV_CNT_PAYMENT_MAX
- PREV_CNT_PAYMENT_SUM

Total features: 34
